# Imports

In [8]:
import spacy
import pandas as pd
from collections import defaultdict
from tqdm import tqdm  # Correct import

df = pd.read_csv('../../Data/2. IntermediateData/df_binarized_hard.csv')

In [ ]:
# Initialize spaCy with optimizations
try:
    nlp = spacy.load("en_core_web_sm", disable=["lemmatizer", "ner"])
except OSError:
    print("Downloading spaCy model...")
    import spacy.cli
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm", disable=["lemmatizer", "ner"])

nlp.add_pipe("sentencizer")

# Take first 1000 rows with original index
sample_df = df.iloc[:1000].copy()

def analyze_with_progress(df_chunk):
    """Process DataFrame chunk with tqdm"""
    results = []
    # Create tqdm iterator explicitly
    tqdm_iterator = tqdm(df_chunk.iterrows(), total=len(df_chunk), desc="Analyzing dialogues")
    
    for _, row in tqdm_iterator:
        doc = nlp(row["text"])
        subjects = objects = 0
        
        for token in doc:
            # Female reference check
            is_female = (
                token.text.lower() in {"she", "her", "hers", "woman", "girl" } or
                (token.text.lower() == "i" and row["gender"] == "f")
            )
            
            if is_female:
                if token.dep_ in ("nsubj", "nsubjpass"):
                    subjects += 1
                elif token.dep_ in ("dobj", "iobj", "pobj"):
                    objects += 1
        
        results.append({"female_subjects": subjects, "female_objects": objects})
    
    return pd.DataFrame(results)

# Process with progress bar
print("🔍 Analyzing gender references in dialogues...")
result_df = analyze_with_progress(sample_df)

# Combine results
sample_df = pd.concat([sample_df, result_df], axis=1)

# Display results
print("\n📊 Results:")
print(f"Total female subjects: {sample_df['female_subjects'].sum()}")
print(f"Total female objects: {sample_df['female_objects'].sum()}\n")

print("By gender category:")
print(sample_df.groupby("gender")[["female_subjects", "female_objects"]].sum())

print("\n✅ Analysis complete!")

🔍 Analyzing gender references in dialogues...


Analyzing dialogues: 100%|██████████| 1000/1000 [00:01<00:00, 513.56it/s]



📊 Results:
Total female subjects: 199
Total female objects: 37

By gender category:
        female_subjects  female_objects
gender                                 
f                   166               3
m                    33              34

✅ Analysis complete!


In [10]:
sample_df

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,speaker_id,character_name,movie_id.1,title,...,id,imdbid,bechdel_score,imdb_score,numVotes,runtimeMinutes,genres,oscar,female_subjects,female_objects
0,L1045,L1044,They do not!,u0,m0,L1044,u0,BIANCA,m0,10 things i hate about you,...,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0,0
1,L1044,L1044,They do to!,u2,m0,NaN,u2,CAMERON,m0,10 things i hate about you,...,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0,0
2,L985,L984,I hope so.,u0,m0,L984,u0,BIANCA,m0,10 things i hate about you,...,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,1,0
3,L984,L984,She okay?,u2,m0,NaN,u2,CAMERON,m0,10 things i hate about you,...,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,1,0
4,L925,L924,Let's go.,u0,m0,L924,u0,BIANCA,m0,10 things i hate about you,...,374,147800,1,7.4,424659,97,"Comedy,Drama,Romance",0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,L5084,L5083,You stay with me...,u67,m4,L5083,u67,CATES,m4,48 hrs.,...,5355,83511,0,6.9,90341,96,"Action,Comedy,Crime",0,0,0
996,L5083,L5083,Bullshit. Then i'm staying with the money.,u71,m4,NaN,u71,HAMMOND,m4,48 hrs.,...,5355,83511,0,6.9,90341,96,"Action,Comedy,Crime",0,0,0
997,L5077,L5076,Right. if you ever switch from armed robbery t...,u67,m4,L5076,u67,CATES,m4,48 hrs.,...,5355,83511,0,6.9,90341,96,"Action,Comedy,Crime",0,0,0
998,L5076,L5076,"That was in style a couple years back, man.",u71,m4,NaN,u71,HAMMOND,m4,48 hrs.,...,5355,83511,0,6.9,90341,96,"Action,Comedy,Crime",0,0,0


In [ ]:
import spacy
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

# Initialize spaCy with optimizations
try:
    nlp = spacy.load("en_core_web_sm", disable=["lemmatizer", "ner"])
except OSError:
    print("Downloading spaCy model...")
    import spacy.cli
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm", disable=["lemmatizer", "ner"])

nlp.add_pipe("sentencizer")

# ===== EXPANDED GENDERED REFERENCES DICTIONARY =====
GENDERED_TERMS = {
    # Female references
    "female": {
        "pronouns": {"she", "her", "hers", "herself"},
        "nouns": {"woman", "women", "girl", "girls", "lady", "ladies", "female", "females", "mother", "daughter", "sister", "actress"}
    },
    # Male references
    "male": {
        "pronouns": {"he", "him", "his", "himself"},
        "nouns": {"man", "men", "boy", "boys", "gentleman", "gentlemen", "male", "males", "father", "son", "brother", "actor"}
    },
    # Neutral/inclusive references
    "neutral": {
        "pronouns": {"they", "them", "their", "theirs", "themself", "themselves"},
        "nouns": {"person", "people", "child", "children", "individual", "actor", "doctor"}  # Gender-neutral professional terms
    }
}

# Take first 1000 rows for demonstration (replace with full_df for complete analysis)
sample_df = df.copy()

def analyze_with_progress(df_chunk):
    """Process DataFrame with expanded gender analysis"""
    results = []
    tqdm_iterator = tqdm(df_chunk.iterrows(), total=len(df_chunk), desc="Analyzing dialogues")
    
    for _, row in tqdm_iterator:
        doc = nlp(row["text"])
        counts = defaultdict(int)
        
        for token in doc:
            # Check all gender categories
            for gender_category, terms in GENDERED_TERMS.items():
                # Check pronouns and nouns
                if (token.text.lower() in terms["pronouns"]) or \
                   (token.text.lower() in terms["nouns"]) or \
                   (token.text.lower() == "i" and row["gender"].lower() == gender_category[0]):  # Handle "I" based on speaker gender
                    
                    # Subject/object analysis
                    if token.dep_ in ("nsubj", "nsubjpass"):
                        counts[f"{gender_category}_subjects"] += 1
                    elif token.dep_ in ("dobj", "iobj", "pobj"):
                        counts[f"{gender_category}_objects"] += 1
        
        results.append(counts)
    
    return pd.DataFrame(results)

# Process with progress bar
print("🔍 Analyzing expanded gender references...")
result_df = analyze_with_progress(sample_df)

# Combine results (fill NaN with 0 for missing categories)
sample_df = pd.concat([sample_df, result_df.fillna(0)], axis=1)

# Display results
print("\n📊 Results by Gender Category:")
gender_categories = ["female", "male", "neutral"]
for category in gender_categories:
    print(f"\n{category.capitalize()} References:")
    print(f"Subjects: {sample_df[f'{category}_subjects'].sum():>6}")
    print(f"Objects:  {sample_df[f'{category}_objects'].sum():>6}")

print("\n📈 Subject/Object Ratios:")
for category in gender_categories:
    total = sample_df[[f'{category}_subjects', f'{category}_objects']].sum().sum()
    if total > 0:
        ratio = sample_df[f'{category}_subjects'].sum() / total
        print(f"{category.capitalize()}: {ratio:.1%} subject references")

print("\n✅ Analysis complete!")

🔍 Analyzing expanded gender references...


Analyzing dialogues: 100%|██████████| 137062/137062 [04:58<00:00, 459.80it/s]



📊 Results by Gender Category:

Female References:
Subjects: 29025.0
Objects:  3743.0

Male References:
Subjects: 55602.0
Objects:  5942.0

Neutral References:
Subjects: 8607.0
Objects:  3533.0

📈 Subject/Object Ratios:
Female: 88.6% subject references
Male: 90.3% subject references
Neutral: 70.9% subject references

✅ Analysis complete!


In [18]:
import scipy.stats as stats
import numpy as np
from statsmodels.stats.proportion import proportions_ztest  # Correct import

# Your results
counts = np.array([29025, 55602])    # Female and male subjects
nobs = np.array([29025+3743, 55602+5942])  # Total references per gender

# 1. Two-proportion z-test
z_score, p_value = proportions_ztest(counts, nobs)
print(f"Female vs Male Subject Ratios:")
print(f"Z = {z_score:.2f}, p = {p_value:.20f} {'***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'}")

# 2. Effect size (Cohen's h)
female_ratio = counts[0]/nobs[0]
male_ratio = counts[1]/nobs[1]
h = 2*(np.arcsin(np.sqrt(female_ratio)) - np.arcsin(np.sqrt(male_ratio)))
print(f"Effect Size (Cohen's h): {abs(h):.3f}")

# 3. Install statsmodels if needed
try:
    from statsmodels.stats.proportion import proportions_ztest
except ImportError:
    print("\nInstalling required package...")
    import sys
    !{sys.executable} -m pip install statsmodels
    from statsmodels.stats.proportion import proportions_ztest

Female vs Male Subject Ratios:
Z = -8.52, p = 0.00000000000000001650 ***
Effect Size (Cohen's h): 0.058


In [14]:
sample_df

,utterance_id,conversation_id,text,speaker,movie_id,reply_to,speaker_id,character_name,movie_id.1,title,...,numVotes,runtimeMinutes,genres,oscar,neutral_subjects,female_subjects,male_subjects,male_objects,neutral_objects,female_objects
0,L1045,L1044,They do not!,u0,m0,L1044,u0,BIANCA,m0,10 things i hate about you,...,424659,97,"Comedy,Drama,Romance",0,1.0,0.0,0.0,0.0,0.0,0.0
1,L1044,L1044,They do to!,u2,m0,NaN,u2,CAMERON,m0,10 things i hate about you,...,424659,97,"Comedy,Drama,Romance",0,1.0,0.0,0.0,0.0,0.0,0.0
2,L985,L984,I hope so.,u0,m0,L984,u0,BIANCA,m0,10 things i hate about you,...,424659,97,"Comedy,Drama,Romance",0,0.0,1.0,0.0,0.0,0.0,0.0
3,L984,L984,She okay?,u2,m0,NaN,u2,CAMERON,m0,10 things i hate about you,...,424659,97,"Comedy,Drama,Romance",0,0.0,1.0,0.0,0.0,0.0,0.0
4,L925,L924,Let's go.,u0,m0,L924,u0,BIANCA,m0,10 things i hate about you,...,424659,97,"Comedy,Drama,Romance",0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137057,L665991,L665987,"I'm sorry, sir. We only seat by reservation.",u9021,m615,L665990,u9021,MAITRE D',m615,young frankenstein,...,176404,106,Comedy,0,0.0,1.0,0.0,0.0,0.0,0.0
137058,L665990,L665987,Food!!,u9023,m615,L665989,u9023,MONSTER,m615,young frankenstein,...,176404,106,Comedy,0,0.0,0.0,0.0,0.0,0.0,0.0
137059,L665989,L665987,Do you have a reservation?,u9021,m615,L665988,u9021,MAITRE D',m615,young frankenstein,...,176404,106,Comedy,0,0.0,0.0,0.0,0.0,0.0,0.0
137060,L665988,L665987,Food!,u9023,m615,L665987,u9023,MONSTER,m615,young frankenstein,...,176404,106,Comedy,0,0.0,0.0,0.0,0.0,0.0,0.0
